In [63]:
from typing import TypedDict, Annotated, List, Union
import operator
from langchain_core.messages import AIMessage
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.types import Command, interrupt
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import MemorySaver
import sqlite3
from langchain.tools import tool
from langchain_tavily import TavilySearch
import datetime

In [64]:
# sqliite_saver=sqlite3.connect("SQLite_Database",check_same_thread=False)
# memory=SqliteSaver(sqliite_saver)
memory=MemorySaver()

In [65]:
class AgentState(TypedDict):
    input:str
    tool_result:Annotated[List, operator.add]
    errors:Union[Annotated[List,operator.add],None]
    chat_history:Annotated[List, operator.add]
    output:Union[str,None]

In [66]:
tools=[]

In [67]:
@tool
def get_system_time(format:str="%Y:%m:%d %H-%m-%S"):
    """Returns the system time in the specified format"""
    current_time = datetime.datetime.now()
    formatted_time=current_time.strftime(format)
    return formatted_time

search_tool=TavilySearch(search_depth="basic")

tools=[get_system_time,search_tool]

In [68]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

llm_with_tools=llm.bind_tools(tools=tools)

prompt_template=ChatPromptTemplate.from_messages([
    ("system","You are an AI Agent built using langgraph. You should not use the tools if you can answer the question using the information below\n"
    "Chat history: \n"
    "{chat_history}\n"
    "Tool results: This is the information you have collected by running the tools. Use this to answer the question insted of using tools.\n "
    "{tool_result}\n"
    "For the current process the following error was encountered by the llm, address this: \n"
    "{errors}"),
    ("human","{input}"),
    ("system","Use the Chat history, Tool results and error given above to answer the question. Do not use tools")
])

In [69]:
responder_chain=prompt_template | llm_with_tools

def responder_node(state:AgentState):
    max_try=3
    current_state=state
    for i in range(max_try):
        try:
            result=responder_chain.invoke(current_state)
            if isinstance(result,AIMessage) and hasattr(result,"tool_calls") and len(result.tool_calls)>0:
                return Command(
                    goto="interrupt",
                    update={"output":result}
                )
            else:
                # print(f"*Secondtime*********{result.content}***********")
                return Command(
                    goto="END",
                    update={"output":result.content[0]["text"],"chat_history":[current_state["input"],result.content[0]["text"]]},
                    )
        except Exception as e:
            last_error=e
            print(f"The following error occured during the invoke on the {i} attempt: {last_error}")
            error_msg=f"The following error occured during the last invoke :\n{last_error}"
            current_state["errors"]=[error_msg]

    result=f"The following error persists on the {max_try} invoke :\n{last_error}"
    return Command(
        goto="END",
        update={"errors": current_state.get("errors", None) + [result]}
    )

def tool_node(state:AgentState):
    result=[]
    errors=[]
    last_AI_Message=state["output"]
    for tool in last_AI_Message.tool_calls:
        # print(last_AI_Message)
        # print(tool)
        # print(tool["name"])
        tool_name=tool["name"]
        tool_args=tool["args"]
        tool_to_be_executed=next((t for t in tools if t.name==tool_name),None)
        
        if tool_to_be_executed:
            tool_result=tool_to_be_executed.invoke(tool_args)
            # print(tool_result)
        else:
            errors.append(f"The tool that was called was {tool_name} which is not prestnt in the list of tools provided."
                        "Only use the the following tools:\n{tools}")

        result.append((tool_name,tool_result))
    
    if errors:
        # print("*********EXECUTED_ERRORS******")
        return Command(
            goto="responder",
            update={"tool_result":result, "errors":errors}
        )
    else:
        # print("*********EXECUTED_NON_ERRORS******")
        return Command(
            goto="responder",
            update={"tool_result":result}
        )

def interrupt_node(state:AgentState):
    human_response=interrupt("Do you want to call the tool? Press (Y/N):")
    if human_response=="Y":
        return Command(
            goto="tool_executer"
        )
    else:
        return Command(
            goto=END
        )

In [70]:
graph=StateGraph(AgentState)
graph.add_node("responder",responder_node)
graph.add_node("tool_executer",tool_node)
graph.add_node("interrupt",interrupt_node)

graph.set_entry_point('responder')

app=graph.compile(memory)

config={"configurable":{"thread_id":3}}

In [71]:
query={
    "input":"What is the current system time?",
    "tool_result":[],
    "errors":None,
    "chat_history":[],
    "output":None
}

# answer=app.invoke(Command(resume="Y"),query,config=config)

answer=app.invoke(query,config=config)

In [72]:
answer

{'input': 'What is the current system time?',
 'tool_result': [],
 'errors': None,
 'chat_history': [],
 'output': AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_system_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'a377c4bf-91bc-4ab7-b6a5-4deec0bc3e6e': 'CooEAQw51sc8RjiI8dVWm5xt2dx1+R3vt3+1XC2okEnfzAbugvoeKSC2jy7oMR3GKarMWVBNf8aLHEq6q4qqMSGBuqXfcH29QnKf84RA+U0JqxT3pv8xnJO/+jrlItzgq+GQ0+F+xpAqrSS//TTFfg3eA1QB6ej8Lk8DMoHaIiXkiKRJXHaYIKdAMEr+T4r0JlaTqHk9Hoi+Hpu4URZ87jvqN6gJc/ySg2jj8v069pTeN7Uo7S3yQcCoPMAo6CEjTFDOUSFOxHKtoyNv72qoUH3TR7H/j4H3N0WJxA9VfQm4jd+rGRktg68dIACesA6AY/e5D3tlM1hrwBrpzoeT6tZrVhJ5KRD6BNVJ1jwoVA63+lMKk0waz03XJ+DSytVrNrLJMhUxX2P80UZ6Z5T8HsIdyuEgOe2o7tx/4DWCJUGiSnDLGda5RktwoFSiZolW9vWG1xyO9yWTcxUMc4dU+BAsanEGj4+j41m1wLsmgElb8b9g1MS8wSkA8DYU/tpbozSOdoECRyAM372QAzwv1EPH1Vwl7OsJI+i0Eze9erWPaMOfpevbf2MGdYTFA5knrKWW+GKllvnadSaHKmfp9oj0u4hd+6mvumEsugeEvnAj2inD2AoR2BZqzHswcewftQakF7MFsbmHLP04oS5kyxijCJMNzz/Cg8mbdfKejpd8obiBzl

In [73]:
print(app.get_state(config).next)

('interrupt',)


In [74]:
answer=app.invoke(Command(resume="Y"),config=config)

Task responder with path ('__pregel_pull', 'responder') wrote to unknown channel branch:to:END, ignoring it.


In [75]:
answer

{'input': 'What is the current system time?',
 'tool_result': [('get_system_time', '2026:06:26 01-06-26')],
 'errors': None,
 'chat_history': ['What is the current system time?',
  'The current system time is 2026-06-26 01:06:26.'],
 'output': 'The current system time is 2026-06-26 01:06:26.'}

In [76]:
print(app.get_state(config).next)

()
